# Thực nghiệm CV–Job Matching với ba tín hiệu

Notebook này là **entry point chính** cho giao thức trong `EXPERIMENTS.md`.

- Core 1: `semantic + skill + experience → Dawid–Skene weak probability`.
- Core 2: so sánh pointwise, pairwise và listwise Learning-to-Rank.
- Gold được chia theo query thành 4 job validation và 8 job test.
- Gold-test chỉ được mở sau khi threshold, formulation, seed và preprocessing đã khóa.
- Không gọi weak probability hoặc author-annotated benchmark là nhãn tuyển dụng thật.

**Giả thuyết chính:** LTR ba tín hiệu vượt công thức trọng số thủ công trên nDCG@5 của Gold-test; chỉ ủng hộ khi paired-bootstrap CI 95% của chênh lệch không chứa 0.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display, Markdown
from src.experiments.runner import ThreeSignalExperiment

SMOKE = True  # Đổi thành False cho authoritative full run duy nhất.
CONFIG = ROOT / "configs" / "experiment_3signal.yaml"
experiment = ThreeSignalExperiment(CONFIG, smoke=SMOKE)
print({"mode": "smoke" if SMOKE else "full", "output": str(experiment.output_dir)})


{'mode': 'smoke', 'output': 'D:\\NCKH\\paper-2026\\results\\three_signal\\smoke'}


## 1. Audit dữ liệu và Gold identity

Kiểm tra schema thật, dòng trùng, phân phối Gold và tạo manifest 4/8 theo job.


In [2]:
gold_validation, raw_audit = experiment.audit_data_and_gold()
display(raw_audit)
display(experiment.gold_manifest)
display(experiment.gold["relevance"].value_counts().sort_index().rename("count"))


{'jobs_raw_rows': 14634,
 'jobs_columns': 19,
 'jobs_unique': 14634,
 'jobs_exact_duplicates': 0,
 'jobs_missing_cells': 0,
 'candidates_raw_rows': 3983,
 'candidates_columns': 14,
 'candidates_unique': 3191,
 'candidates_exact_duplicates': 792,
 'candidates_missing_cells': 0}

{'protocol': 'query-disjoint-gold-4-validation-8-test',
 'seed': 42,
 'validation_jobs': ['JOB_14', 'JOB_51', 'JOB_71', 'JOB_74'],
 'test_jobs': ['JOB_01',
  'JOB_02',
  'JOB_20',
  'JOB_21',
  'JOB_23',
  'JOB_29',
  'JOB_52',
  'JOB_60'],
 'validation_pairs': 29,
 'test_pairs': 71}

relevance
0    30
1    35
2    29
3     6
Name: count, dtype: int64

## 2. Candidate lists, split theo job và train-only preprocessing

Mọi job/CV đã xuất hiện trong Gold đều bị loại khỏi weak-label pool trước sampling và fit.


In [3]:
development_counts = experiment.prepare_development_data()
display(development_counts)
display(experiment.feature_pipeline.manifest())
assert set(experiment.feature_pipeline.manifest()["feature_columns"]) == {"s_sem", "s_skill", "s_exp"}


{'sampled_jobs': 36,
 'sampled_candidates': 80,
 'sampled_pairs': 1080,
 'train_pairs': 750,
 'validation_pairs': 150,
 'development_test_pairs': 180}

{'feature_columns': ['s_sem', 's_skill', 's_exp'],
 'baseline_components': ['baseline_location',
  'baseline_skill',
  'baseline_experience',
  'baseline_role',
  'baseline_description'],
 'fit_job_ids': ['JOB_10764',
  'JOB_10950',
  'JOB_11097',
  'JOB_11854',
  'JOB_11973',
  'JOB_12097',
  'JOB_181',
  'JOB_2947',
  'JOB_3311',
  'JOB_3663',
  'JOB_369',
  'JOB_3850',
  'JOB_3962',
  'JOB_4111',
  'JOB_4758',
  'JOB_4901',
  'JOB_5489',
  'JOB_5933',
  'JOB_8072',
  'JOB_848',
  'JOB_8814',
  'JOB_8941',
  'JOB_9537',
  'JOB_9693',
  'JOB_9765'],
 'fit_candidate_ids': ['CV_046',
  'CV_1186',
  'CV_1237',
  'CV_1393',
  'CV_1406',
  'CV_1432',
  'CV_1449',
  'CV_1471',
  'CV_1502',
  'CV_169',
  'CV_201',
  'CV_206',
  'CV_2376',
  'CV_2404',
  'CV_2418',
  'CV_245',
  'CV_2469',
  'CV_2473',
  'CV_2480',
  'CV_2503',
  'CV_2531',
  'CV_2579',
  'CV_259',
  'CV_2600',
  'CV_261',
  'CV_2623',
  'CV_2677',
  'CV_2741',
  'CV_2745',
  'CV_278',
  'CV_281',
  'CV_2812',
  'CV_2822',
  

## 3. Ba labeling functions và Dawid–Skene

Ngưỡng 25/75 chỉ fit trên development-train. Bảng chất lượng dùng Gold-validation; luật 3/3 là đối chứng.


In [4]:
label_quality, lf_statistics, lf_pair_diagnostics = experiment.fit_weak_supervision()
display(lf_statistics)
display(lf_pair_diagnostics)
display(label_quality)
print("Selected posterior threshold:", experiment.posterior_threshold)


,lf,coverage,positive_rate,negative_rate,abstain_rate
0,lf_sem,0.501333,0.250667,0.250667,0.498667
1,lf_skill,0.966667,0.001333,0.965333,0.033333
2,lf_exp,0.520000,0.268000,0.252000,0.480000


,lf_a,lf_b,joint_coverage,agreement,conflict,spearman
0,lf_sem,lf_skill,0.481333,0.498615,0.241333,-0.010280
1,lf_sem,lf_exp,0.246667,0.572973,0.105333,0.070484
2,lf_skill,lf_exp,0.504000,0.492063,0.256000,0.046562


,method,precision,recall,coverage,n_covered,threshold,condition_passed
0,strict_3_of_3,0.0,0.0,0.034483,1,NaN,NaN
1,dawid_skene,0.0,0.0,1.000000,29,0.3,False


Selected posterior threshold: 0.3


## 4. Pointwise, pairwise và listwise trên Gold-validation

Ba formulation dùng cùng scorer tuyến tính và đúng ba feature. Hyperparameter/early stopping dùng weak-validation objective; Gold-validation chỉ quyết định formulation.


In [5]:
formulation_summary = experiment.train_and_select_formulation()
display(formulation_summary)
print("Selected formulation:", experiment.selected_formulation)


,formulation,ndcg@5,ndcg@10,mrr,selected
0,listwise,0.525337,0.634094,0.500,False
1,pairwise,0.465264,0.572208,0.300,False
2,pointwise,0.626679,0.666032,0.375,True


Selected formulation: pointwise


## 5. Khóa protocol trước Gold-test

Cell này ghi threshold, formulation, feature/weak-label state và model metadata. Không có Gold-test score trước thời điểm này.


In [6]:
protocol_lock = experiment.lock_protocol()
display(protocol_lock)
assert protocol_lock["locked"] and not protocol_lock["test_opened"]


{'locked': True,
 'selected_threshold': 0.3,
 'selected_formulation': 'pointwise',
 'metadata': {'seeds': [42],
  'feature_manifest': {'feature_columns': ['s_sem', 's_skill', 's_exp'],
   'baseline_components': ['baseline_location',
    'baseline_skill',
    'baseline_experience',
    'baseline_role',
    'baseline_description'],
   'fit_job_ids': ['JOB_10764',
    'JOB_10950',
    'JOB_11097',
    'JOB_11854',
    'JOB_11973',
    'JOB_12097',
    'JOB_181',
    'JOB_2947',
    'JOB_3311',
    'JOB_3663',
    'JOB_369',
    'JOB_3850',
    'JOB_3962',
    'JOB_4111',
    'JOB_4758',
    'JOB_4901',
    'JOB_5489',
    'JOB_5933',
    'JOB_8072',
    'JOB_848',
    'JOB_8814',
    'JOB_8941',
    'JOB_9537',
    'JOB_9693',
    'JOB_9765'],
   'fit_candidate_ids': ['CV_046',
    'CV_1186',
    'CV_1237',
    'CV_1393',
    'CV_1406',
    'CV_1432',
    'CV_1449',
    'CV_1471',
    'CV_1502',
    'CV_169',
    'CV_201',
    'CV_206',
    'CV_2376',
    'CV_2404',
    'CV_2418',
    'CV

## 6. Mở Gold-test đúng một lần: kết quả chính và hai ablation

Hệ thống đầy đủ được so với công thức thủ công. Hai ablation duy nhất là bỏ label model và bỏ LTR.


In [7]:
gold_test_summary, bootstrap_results, gold_test_per_job = experiment.evaluate_gold_test_once()
display(gold_test_summary)
display(bootstrap_results)


,system,ndcg@5,ndcg@10,mrr
0,ablation_direct_probability,0.766384,0.811049,0.629167
1,ablation_mean_signal_ltr,0.637986,0.720582,0.424107
2,manual_score_h,0.805189,0.832431,0.566667
3,selected_ltr,0.637986,0.720582,0.424107


,comparison,baseline,proposed,metric,n_jobs,mean_delta,ci_95_low,ci_95_high,supports_improvement
0,main,manual_score_h,selected_ltr,ndcg@5,8,-0.167204,-0.406125,0.069097,False
1,main,manual_score_h,selected_ltr,ndcg@10,8,-0.111849,-0.268321,0.052008,False
2,main,manual_score_h,selected_ltr,mrr,8,-0.142560,-0.440625,0.172917,False
3,core_1,ablation_mean_signal_ltr,selected_ltr,ndcg@5,8,0.000000,0.000000,0.000000,False
4,core_1,ablation_mean_signal_ltr,selected_ltr,ndcg@10,8,0.000000,0.000000,0.000000,False
5,core_1,ablation_mean_signal_ltr,selected_ltr,mrr,8,0.000000,0.000000,0.000000,False
6,core_2,ablation_direct_probability,selected_ltr,ndcg@5,8,-0.128399,-0.395048,0.193886,False
7,core_2,ablation_direct_probability,selected_ltr,ndcg@10,8,-0.090467,-0.291795,0.138930,False
8,core_2,ablation_direct_probability,selected_ltr,mrr,8,-0.205060,-0.524256,0.137656,False


## 7. Kết luận có điều kiện và xuất artifact

Kết luận được sinh từ CI của nDCG@5, không chọn câu chuyện sau khi xem số.


In [8]:
run_manifest = experiment.finalize()
display(run_manifest)
display(Markdown(f"**Kết luận smoke/full run:** {run_manifest['conclusion']}"))
print("Artifacts:", experiment.output_dir)


{'protocol_version': 'three-signal-2026-08-24',
 'mode': 'smoke',
 'stage': 'complete',
 'selected_formulation': 'pointwise',
 'selected_posterior_threshold': 0.3,
 'conclusion': 'NOT SUPPORTED: the nDCG@5 confidence interval contains or touches zero.',
 'counts': {'gold_validation_jobs': 4, 'gold_test_jobs': 8, 'seeds': 1},
 'input_hashes': {'jobs': '881b6725bf79cc44441225443cd33580bd3726281b09a8d7a8dd86622bfbc69a',
  'candidates': 'e620c5a0c577af130f0a918bf40dfc4360c693bd8e95e8f44cc50107a92a235c',
  'gold': '7b1fa84573b504327d3cafb95f3293f9a981b90bc51d93f89ed3cee3cfde5d94'}}

**Kết luận smoke/full run:** NOT SUPPORTED: the nDCG@5 confidence interval contains or touches zero.

Artifacts: D:\NCKH\paper-2026\results\three_signal\smoke


## Hạn chế khi diễn giải

Gold chỉ có 12 query và một annotator; CI có thể rộng. Dataset không có click/apply/hiring outcome. Label model giả định độc lập có điều kiện giữa ba nguồn dù semantic có thể chứa thông tin skill/experience. Smoke mode chỉ kiểm tra tích hợp và **không phải paper numbers**.
